In [1]:
from dataclasses import dataclass
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, save_and_log_checkpoint, load_training_checkpoint
import mlflow
import torch
import json
from mamba_ssm import Mamba2
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKMamba2Adapter
from model_classes import WaitKMamba2MT, SimulMamba2Config, WaitKTransformerMT, SimulTransformerConfig

/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_mamba2_distillation")

<Experiment: artifact_location='/mlflow/artifacts/2', creation_time=1779451055489, experiment_id='2', last_update_time=1779451055489, lifecycle_stage='active', name='simulmt_waitk_mamba2_distillation', tags={}, trace_location=None, workspace='default'>

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [4]:
mamba2_config = SimulMamba2Config(
    vocab_size=len(tokenizer),

    d_model=512,
    num_layers=12,

    d_state=128,
    d_conv=4,
    expand=2,
    headdim=64,
    ngroups=1,

    dropout=0.1,

    max_source_len=64,
    max_target_len=64,

    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,

    tie_embeddings=True,
    separate_source_target_embeddings=True,
)

mamba2 = WaitKMamba2MT(mamba2_config)
count_parameters(mamba2)

Total parameters:     283,070,528
Trainable parameters: 283,070,528
Frozen parameters:    0


{'total': 283070528, 'trainable': 283070528, 'frozen': 0}

In [5]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [6]:
train_cfg = TrainConfig(
    epochs=8,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=1e-4,
    use_amp=True,
    save_every_steps=100000
)

In [7]:
train_waitk_student(
    student=mamba2,
    train_dataset=dataset,
    model_cfg=mamba2_config,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="mamba",
    compile_model=False,
    save_latest=False
)

epoch 1/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 2/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 3/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 4/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 5/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 6/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 7/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 8/8:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/2/runs/e3576becbb304f528584daf0fd3c7e1d
🧪 View experiment at: http://localhost:5000/#/experiments/2


WaitKMamba2MT(
  (src_embedding): Embedding(256204, 512, padding_idx=1)
  (tgt_embedding): Embedding(256204, 512, padding_idx=1)
  (position_embedding): Embedding(132, 512)
  (segment_embedding): Embedding(3, 512)
  (dropout): Dropout(p=0.1, inplace=False)
  (layers): ModuleList(
    (0-11): 12 x Mamba2Block(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mixer): Mamba2(
        (in_proj): Linear(in_features=512, out_features=2320, bias=False)
        (conv1d): Conv1d(1280, 1280, kernel_size=(4,), stride=(1,), padding=(3,), groups=1280)
        (act): SiLU()
        (norm): RMSNorm()
        (out_proj): Linear(in_features=1024, out_features=512, bias=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=256204, bias=False)
)

In [3]:
mamba2 = load_training_checkpoint("./models/mamba.pt", SimulMamba2Config, WaitKMamba2MT)[0]

In [4]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

In [5]:
adapter = WaitKMamba2Adapter(
    model=mamba2,
    tokenizer=tokenizer,
    name="mamba2_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(
        use_comet=True,
        comet_model_name="Unbabel/wmt22-comet-da",
        comet_batch_size=512,
    ),
)

for k in [3, 5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63,
        dataset_fraction=0.1
    )
    
    print(result.metrics)
    
    with open(f"./metrics/mamba_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Validating mamba2_wait_k, wait_k=3:   0%|          | 0/31 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/

{'BLEU': 27.33954323527312, 'chrF++': 48.33307336856948, 'TER': 68.24947392339534, 'COMET': 0.7202204866017241, 'AP': 0.6565518551917635, 'AL': 5.150947258796486, 'DAL': 5.910790855085973, 'LAAL': 5.512756663223482, 'ATD_text': 3.708761406233232, 'total_time_sec': 48.871246378999786, 'ms_per_sentence': 1.5794979599560384, 'target_tokens_per_sec': 21328.08301872764, 'source_tokens_per_sec': 17783.74943131106, 'first_token_latency_sec': 1.453538867013051, 'peak_gpu_memory_mb': 10382.99267578125, 'dataset_fraction': 0.1, 'eval_dataset_size': 30941, 'full_dataset_size': 309410, 'generation_total_time_sec': 44.54824307998206, 'generation_ms_per_sentence': 1.4397803264271374, 'generation_target_tokens_per_sec': 23397.78020265799}


...

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 61/61 [02:39<00:00, 

{'BLEU': 27.757488825244888, 'chrF++': 48.09262540914466, 'TER': 66.44648837691783, 'COMET': 0.7250371837066173, 'AP': 0.8062999644425944, 'AL': 10.609161033742836, 'DAL': 11.451786069829321, 'LAAL': 9.497909199909934, 'ATD_text': 7.420363346341241, 'total_time_sec': 48.44514869699924, 'ms_per_sentence': 1.5657266635531897, 'target_tokens_per_sec': 21014.343590260443, 'source_tokens_per_sec': 17940.16580351283, 'first_token_latency_sec': 1.4455783849710349, 'peak_gpu_memory_mb': 10487.00048828125, 'dataset_fraction': 0.1, 'eval_dataset_size': 30941, 'full_dataset_size': 309410, 'generation_total_time_sec': 44.3026432350016, 'generation_ms_per_sentence': 1.4318426435797678, 'generation_target_tokens_per_sec': 22979.283529423552}


Validating mamba2_wait_k, wait_k=11:   0%|          | 0/31 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 61/61 [02:39<00:00, 

{'BLEU': 27.115029314711006, 'chrF++': 47.35264307551667, 'TER': 67.19396172730954, 'COMET': 0.7175321077422809, 'AP': 0.8449848338353481, 'AL': 12.41675980186455, 'DAL': 13.24477256499056, 'LAAL': 10.57638553062083, 'ATD_text': 8.4571107228928, 'total_time_sec': 48.401406837998366, 'ms_per_sentence': 1.5643129452182658, 'target_tokens_per_sec': 20971.725127690064, 'source_tokens_per_sec': 17956.378890162487, 'first_token_latency_sec': 1.4439987763952449, 'peak_gpu_memory_mb': 10278.98486328125, 'dataset_fraction': 0.1, 'eval_dataset_size': 30941, 'full_dataset_size': 309410, 'generation_total_time_sec': 44.258897743995476, 'generation_ms_per_sentence': 1.4304288078599747, 'generation_target_tokens_per_sec': 22934.61996887872}
